# CS570 Week 4: Lab Class - Joins & Window Functions

**Today's session:** Interactive coding together (not graded)

**Homework:** The separate Lab notebook (graded, due on Canvas)

---

## Rules for Today

1. **Don't run ahead** - we do this together
2. **Predict before running** - write down your prediction, THEN run
3. **Ask questions** - if you're confused, others are too

---

## Part 1: Setup (everyone together)

Run these cells. Wait for checkpoint before continuing.

In [1]:
import os
import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Ensure Spark workers use the same Python as this notebook
# (Fixes version mismatch errors when using virtual environments)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Create SparkSession with all available cores
spark = SparkSession.builder \
    .appName("CS570 Spotify Analysis") \
    .master("local[*]") \
    .getOrCreate()

# Get SparkContext from session (for RDD operations)
sc = spark.sparkContext

print(f"Spark version: {spark.version}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Default parallelism (cores): {sc.defaultParallelism}")

sc.setLogLevel("ERROR")  # Suppress warnings for cleaner output
print(f"\n✓ Spark is ready with {sc.defaultParallelism} cores!")

Spark version: 3.5.3
Python version: 3.11.9
Default parallelism (cores): 16

✓ Spark is ready with 16 cores!


In [5]:
# Load Spotify data
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv("spotify.csv")

row_count = df.count()
print(f"Rows loaded: {row_count:,}")

Rows loaded: 114,000


### ✅ CHECKPOINT

**Call out your row count.** Everyone should have **114,000**.

If you don't, raise your hand.

---

## Part 2: Joins

### 2.1 Create a lookup table

In real data pipelines, you often have:
- A **fact table** (transactions, events, tracks)
- A **dimension/lookup table** (categories, users, genres)

Let's create a small genre lookup table:

In [6]:
# A small lookup table with extra info about some genres
genre_data = [
    ("pop", "Popular Music", "High"),
    ("rock", "Rock Music", "High"),
    ("jazz", "Jazz Music", "Medium"),
    ("classical", "Classical Music", "Medium"),
    ("fake_genre", "Doesn't Exist In Tracks", "Low")
]

genres = spark.createDataFrame(genre_data, ["genre", "full_name", "market_size"])
genres.show()

+----------+--------------------+-----------+
|     genre|           full_name|market_size|
+----------+--------------------+-----------+
|       pop|       Popular Music|       High|
|      rock|          Rock Music|       High|
|      jazz|          Jazz Music|     Medium|
| classical|     Classical Music|     Medium|
|fake_genre|Doesn't Exist In ...|        Low|
+----------+--------------------+-----------+



Notice:
- We have info for **pop, rock, jazz, classical** (these exist in tracks)
- We have **fake_genre** (does NOT exist in tracks)
- We're **missing** many genres that ARE in tracks (hip-hop, electronic, etc.)

---

### 2.2 Prediction: INNER JOIN

I'm about to run:
```python
df.join(...)
```

## ✋ STOP - PREDICT FIRST

**Write your predictions below before running the next cell:**

1. Will we get MORE rows, FEWER rows, or SAME rows as original (114,000)?
   
   → My prediction: Fewer rows

2. Will "fake_genre" appear in the result?
   
   → My prediction: No

3. What about tracks with genre "hip-hop" (not in our lookup table)?
   
   → My prediction: Dropped

In [7]:
# INNER JOIN
inner_result = df.join(genres, df.track_genre == genres.genre, "inner")
print(f"Inner join rows: {inner_result.count():,}")
inner_result.select("track_name", "track_genre", "full_name", "market_size").show(5)


Inner join rows: 4,000
+-----------------+-----------+-------------+-----------+
|       track_name|track_genre|    full_name|market_size|
+-----------------+-----------+-------------+-----------+
|       Oru Naalil|        pop|Popular Music|       High|
|  Vaarayo Vaarayo|        pop|Popular Music|       High|
|Tu Mile Dil Khile|        pop|Popular Music|       High|
|     Annul Maelae|        pop|Popular Music|       High|
|      Yathe Yathe|        pop|Popular Music|       High|
+-----------------+-----------+-------------+-----------+
only showing top 5 rows



### 2.3 Prediction: LEFT JOIN

Now I'll do a LEFT join.

## ✋ STOP - PREDICT FIRST

1. Will we get MORE rows, FEWER rows, or SAME rows as original (114,000)?
   
   → My prediction: Same rows (114000)

2. For tracks with genre "hip-hop" (not in lookup), what will the columns from the lookup table contain?
   
   → My prediction: Null

In [8]:
# LEFT JOIN
left_result = df.join(genres, df.track_genre == genres.genre, "left")
print(f"Left join rows: {left_result.count():,}")
left_result.select("track_name", "track_genre", "full_name", "market_size").show(5)

Left join rows: 114,000
+--------------------+-----------+---------+-----------+
|          track_name|track_genre|full_name|market_size|
+--------------------+-----------+---------+-----------+
|              Comedy|   acoustic|     NULL|       NULL|
|    Ghost - Acoustic|   acoustic|     NULL|       NULL|
|      To Begin Again|   acoustic|     NULL|       NULL|
|Can't Help Fallin...|   acoustic|     NULL|       NULL|
|             Hold On|   acoustic|     NULL|       NULL|
+--------------------+-----------+---------+-----------+
only showing top 5 rows



In [9]:
# Show rows where no match was found (NULLs in lookup columns)
left_result.filter(col("full_name").isNull()) \
    .select("track_name", "track_genre", "full_name", "market_size") \
    .show(5)

+------------------+-----------+---------+-----------+
|        track_name|track_genre|full_name|market_size|
+------------------+-----------+---------+-----------+
|    カワキヲアメク|      anime|     NULL|       NULL|
|        シルエット|      anime|     NULL|       NULL|
|         KICK BACK|      anime|     NULL|       NULL|
|           unravel|      anime|     NULL|       NULL|
|残酷な天使のテーゼ|      anime|     NULL|       NULL|
+------------------+-----------+---------+-----------+
only showing top 5 rows



### 2.4 Data Quality Check: LEFT ANTI

We just used left join + filter for NULL to find non-matches. But there's a cleaner way.

**Question:** What if I *only* want the tracks that don't have a matching genre in the lookup?

This is `left_anti` - rows from left where key does **NOT** exist in right. No NULLs to filter, no extra columns added.

In [10]:
# LEFT ANTI - find tracks with no matching genre info
anti_result = df.join(genres, df.track_genre == genres.genre, "left_anti")
print(f"Tracks with no genre info: {anti_result.count():,}")

Tracks with no genre info: 110,000


In [11]:
# Which genres are we missing info for?
anti_result.select("track_genre").distinct().orderBy("track_genre").show(20)

+-------------+
|  track_genre|
+-------------+
|     acoustic|
|     afrobeat|
|     alt-rock|
|  alternative|
|      ambient|
|        anime|
|  black-metal|
|    bluegrass|
|        blues|
|       brazil|
|    breakbeat|
|      british|
|     cantopop|
|chicago-house|
|     children|
|        chill|
|         club|
|       comedy|
|      country|
|        dance|
+-------------+
only showing top 20 rows



### Summary: Join Types

| Type | Returns |
|------|--------|
| `inner` | Only rows that match both sides |
| `left` | All left rows + matching right (NULL if no match) |
| `left_anti` | Left rows where key NOT in right |
| `left_semi` | Left rows where key exists in right (no right columns) |

---

## Part 3: Window Functions

### The Problem

**Question: Find the most popular track in each genre.**

Try to use a `groupBy`?

In [12]:
#YOUR CODE
max_pop = df.groupBy("track_genre").agg(max("popularity").alias("max_pop"))
max_pop.show(5)

+-----------------+-------+
|      track_genre|max_pop|
+-----------------+-------+
|            anime|     83|
|singer-songwriter|     90|
|             folk|     90|
|        hardstyle|     70|
|              pop|    100|
+-----------------+-------+
only showing top 5 rows



### 3.1 Define the Window

Window functions give us a cleaner way: **add aggregate info without collapsing rows.**

A window has two parts:
- `partitionBy` - what are the groups? (like GROUP BY)
- `orderBy` - how to sort within each group?

In [13]:
# Define window: within each genre, order by popularity (highest first)
w = Window.partitionBy("track_genre").orderBy(desc("popularity"))


**In plain English:** "Within each genre, order tracks by popularity, highest first."

---

### 3.2 Find the Most Popular Track per Genre

Now we add a rank column using `rank().over(w)`, then filter to rank 1:

In [14]:
# Add rank column, look at one genre
ranked = df.withColumn("rnk", rank().over(w))

ranked.filter(col("track_genre") == "pop") \
    .select("track_name", "artists", "track_genre", "popularity", "rnk") \
    .show(10)

+--------------------+--------------------+-----------+----------+---+
|          track_name|             artists|track_genre|popularity|rnk|
+--------------------+--------------------+-----------+----------+---+
|Unholy (feat. Kim...|Sam Smith;Kim Petras|        pop|       100|  1|
|     I'm Good (Blue)|David Guetta;Bebe...|        pop|        98|  2|
| Under The Influence|         Chris Brown|        pop|        96|  3|
|     I Ain't Worried|         OneRepublic|        pop|        96|  3|
|           As It Was|        Harry Styles|        pop|        95|  5|
|       Glimpse of Us|                Joji|        pop|        94|  6|
|     Sweater Weather|   The Neighbourhood|        pop|        93|  7|
|        Another Love|           Tom Odell|        pop|        93|  7|
|Left and Right (F...|Charlie Puth;Jung...|        pop|        92|  9|
|Calm Down (with S...|   Rema;Selena Gomez|        pop|        92|  9|
+--------------------+--------------------+-----------+----------+---+
only s

In [15]:
# Filter to rank 1 = most popular per genre
top_per_genre = ranked.filter(col("rnk") == 1) \
    .select("track_name", "artists", "track_genre", "popularity")

top_per_genre.show(10)


+-----------------+--------------------+-----------+----------+
|       track_name|             artists|track_genre|popularity|
+-----------------+--------------------+-----------+----------+
|          Hold On|    Chord Overstreet|   acoustic|        82|
|   Atrévete-Te-Te|            Calle 13|   afrobeat|        75|
|  Sweater Weather|   The Neighbourhood|   alt-rock|        93|
|  Sweater Weather|   The Neighbourhood|alternative|        93|
|       Apocalypse|Cigarettes After Sex|    ambient|        84|
|        KICK BACK|       Kenshi Yonezu|      anime|        83|
|       Doomswitch|    Make Them Suffer|black-metal|        58|
|         Daylight|          Watchhouse|  bluegrass|        69|
|Seven Nation Army|   The White Stripes|      blues|        84|
|   Move Your Body|       Öwnboss;Sevek|     brazil|        82|
+-----------------+--------------------+-----------+----------+
only showing top 10 rows



**Notice:**
- Every row still exists until we filter (no collapse!)
- Each row has a rank within its genre
- Ties get the same rank (multiple rank 1s possible)

---

### 3.3 What About Top 3?

**Question:** Now find the top 3 most popular tracks in each genre.

## ✋ STOP - PREDICT FIRST

We have 114 genres. If I filter to `rank <= 3`:

1. Exactly how many rows should I get? (assume no ties)
   
   → My prediction: 114 genres x 3 = 342 rows

2. Will I get exactly that many? Why or why not?
   
   → My prediction: Probably More , because rank() creates ties

In [16]:
# Filter to top 3 per genre
top3 = ranked.filter(col("rnk") <= 3)
print(f"Rows with rank <= 3: {top3.count()}")


Rows with rank <= 3: 411


### 3.4 Prediction: row_number() vs rank()

## ✋ STOP - PREDICT FIRST

If I use `row_number()` instead of `rank()`, and filter to `<= 3`:

1. Will I get MORE rows, FEWER rows, or SAME rows?
   
   → My prediction: Fewer (or exactly 342) rows

2. Why?
   
   → My prediction: row_number() NEVER creates ties
   It assigns 1, 2, 3, 4, 5... even for identical values

In [17]:
# Compare row_number() vs rank()
w = Window.partitionBy("track_genre").orderBy(desc("popularity"))

ranked_rn = df.withColumn("rn", row_number().over(w))
ranked_rank = df.withColumn("rnk", rank().over(w))

print(f"row_number() <= 3: {ranked_rn.filter(col('rn') <= 3).count()}")
print(f"rank() <= 3:       {ranked_rank.filter(col('rnk') <= 3).count()}")


row_number() <= 3: 342
rank() <= 3:       411


### Debrief: row_number vs rank vs dense_rank

For popularity values: 100, 100, 90, 80

| Function | Result | Use when... |
|----------|--------|--------------|
| `row_number()` | 1, 2, 3, 4 | You need exactly N rows per group |
| `rank()` | 1, 1, 3, 4 | Ties should share rank, gaps OK |
| `dense_rank()` | 1, 1, 2, 3 | Ties share rank, no gaps |

---

### 3.5 YOUR TURN: 60 Second Challenge ⏱️

**Modify the window to find the LEAST popular track in each genre.**

Hint: You only need to change one thing.

In [18]:
# YOUR TURN: Find the LEAST popular track in each genre
# Change desc("popularity") to asc("popularity")!
w_least = Window.partitionBy("track_genre").orderBy(asc("popularity"))
#                                                    ^^^
#                                              THIS is the change!

least_popular = df.withColumn("rnk", rank().over(w_least)) \
    .filter(col("rnk") == 1) \
    .select("track_name", "artists", "track_genre", "popularity")

least_popular.show(10)


+--------------------+--------------------+-----------+----------+
|          track_name|             artists|track_genre|popularity|
+--------------------+--------------------+-----------+----------+
|    93 Million Miles|          Jason Mraz|   acoustic|         0|
|            Unlonely|          Jason Mraz|   acoustic|         0|
|   Winter Wonderland|          Jason Mraz|   acoustic|         0|
|      If It Kills Me|          Jason Mraz|   acoustic|         0|
|   Winter Wonderland|          Jason Mraz|   acoustic|         0|
|   Winter Wonderland|          Jason Mraz|   acoustic|         0|
|   Winter Wonderland|          Jason Mraz|   acoustic|         0|
|   Winter Wonderland|          Jason Mraz|   acoustic|         0|
|All I Want For Ch...|    Chord Overstreet|   acoustic|         0|
|        Party of One|Brandi Carlile;Sa...|   acoustic|         0|
+--------------------+--------------------+-----------+----------+
only showing top 10 rows



---

## Part 4: lead() and lag() with Chart Data

These let you access values from other rows:
- `lag(col, n)` - value from n rows BEFORE
- `lead(col, n)` - value from n rows AFTER

**Example:** Let's use some Billboard-style chart data to see how songs move up and down the charts.

In [19]:
# Billboard-style chart data
chart_data = [
    # Week 1
    ("2024-01-06", "Lovin On Me", "Jack Harlow", 1),
    ("2024-01-06", "Cruel Summer", "Taylor Swift", 2),
    ("2024-01-06", "Water", "Tyla", 3),
    ("2024-01-06", "Stick Season", "Noah Kahan", 4),
    ("2024-01-06", "Agora Hills", "Doja Cat", 5),
    # Week 2 - positions changed
    ("2024-01-13", "Lovin On Me", "Jack Harlow", 1),
    ("2024-01-13", "Water", "Tyla", 2),        # up from 3
    ("2024-01-13", "Cruel Summer", "Taylor Swift", 3),  # down from 2
    ("2024-01-13", "Agora Hills", "Doja Cat", 4),  # up from 5
    ("2024-01-13", "Stick Season", "Noah Kahan", 5),  # down from 4
    # etc.
]

chart = spark.createDataFrame(chart_data, ["week", "song", "artist", "position"])
chart.show()

+----------+------------+------------+--------+
|      week|        song|      artist|position|
+----------+------------+------------+--------+
|2024-01-06| Lovin On Me| Jack Harlow|       1|
|2024-01-06|Cruel Summer|Taylor Swift|       2|
|2024-01-06|       Water|        Tyla|       3|
|2024-01-06|Stick Season|  Noah Kahan|       4|
|2024-01-06| Agora Hills|    Doja Cat|       5|
|2024-01-13| Lovin On Me| Jack Harlow|       1|
|2024-01-13|       Water|        Tyla|       2|
|2024-01-13|Cruel Summer|Taylor Swift|       3|
|2024-01-13| Agora Hills|    Doja Cat|       4|
|2024-01-13|Stick Season|  Noah Kahan|       5|
+----------+------------+------------+--------+



**Task: For each song, get last week's position and calculate how many spots they moved.**

In [20]:
# Window: for each song, order by week
song_window = Window.partitionBy("song").orderBy("week")

# add new column "last_week"
chart_with_movement = chart \
    .withColumn("last_week", lag("position", 1).over(song_window)) \
    .withColumn("change", col("last_week") - col("position"))

chart_with_movement.orderBy("week", "position").show()

+----------+------------+------------+--------+---------+------+
|      week|        song|      artist|position|last_week|change|
+----------+------------+------------+--------+---------+------+
|2024-01-06| Lovin On Me| Jack Harlow|       1|     NULL|  NULL|
|2024-01-06|Cruel Summer|Taylor Swift|       2|     NULL|  NULL|
|2024-01-06|       Water|        Tyla|       3|     NULL|  NULL|
|2024-01-06|Stick Season|  Noah Kahan|       4|     NULL|  NULL|
|2024-01-06| Agora Hills|    Doja Cat|       5|     NULL|  NULL|
|2024-01-13| Lovin On Me| Jack Harlow|       1|        1|     0|
|2024-01-13|       Water|        Tyla|       2|        3|     1|
|2024-01-13|Cruel Summer|Taylor Swift|       3|        2|    -1|
|2024-01-13| Agora Hills|    Doja Cat|       4|        5|     1|
|2024-01-13|Stick Season|  Noah Kahan|       5|        4|    -1|
+----------+------------+------------+--------+---------+------+



**Notice:** 

- First week has NULL for last_week (no previous data)
- Positive change = moved UP the chart (lower position number is better)
- Water: went from 3 → 2, change = +1 (moved up 1 spot)
- Cruel Summer: went from 2 → 3, change = -1 (dropped 1 spot)

---

## Summary: Window Functions

```python
# 1. Define the window
w = Window.partitionBy("group_col").orderBy("sort_col")

# 2. Apply a function over it
df.withColumn("new_col", some_function().over(w))
```

**Common functions:**
- `rank()`, `dense_rank()`, `row_number()` - ranking
- `lag()`, `lead()` - access other rows
- `sum()`, `avg()` - running totals (with `rowsBetween`)

---

In [21]:
spark.stop()
